In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

api_key = os.environ["OPENAI_API_KEY"]
print("API key loaded successfully")

API key loaded successfully


## 1. Data scraper to scrapes out the data from literature pdfs.

In [2]:
from scraper import get_default_scraper

default_scraper = get_default_scraper(debug=False)

scraper_input = default_scraper.input_schema(
  url = "https://acp.copernicus.org/articles/22/9617/2022/acp-22-9617-2022.pdf"
)

scraped_output = await default_scraper.arun(scraper_input)
scraped_text = scraped_output.content

/Users/thapa24-mbp/Devs/sandbox/accelerated-discovery/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-09 11:40:55.787 | ERROR    | akd._base._base:arun:394 - Error running Crawl4AIWebScraper: Can't parse url with PDF :: https://acp.copernicus.org/articles/22/9617/2022/acp-22-9617-2022.pdf
2025-12-09 11:40:55.787 | ERROR    | akd.tools.scrapers.composite:_arun:92 - Error running Crawl4AIWebScraper
Can't parse url with PDF :: https://acp.copernicus.org/articles/22/9617/2022/acp-22-9617-2022.pdf
2025-12-09 11:40:55.788 | ERROR    | akd._base._base:arun:394 - Error running SimpleWebScraper: Can't parse url with PDF :: https://acp.copernicus.org/articles/22/9617/2022/acp-22-9617-2022.pdf
2025-12-09 11:40:55.788 | ERROR    | akd.tools.scrapers.composite:_arun:92 - Error running SimpleWebScraper


In [3]:
scraped_text

'Atmos. Chem. Phys., 22, 9617–9646, 2022\nhttps://doi.org/10.5194/acp-22-9617-2022\n© Author(s) 2022. This work is distributed under\nthe Creative Commons Attribution 4.0 License.\nReview article\nQuantifying methane emissions from the global scale\ndown to point sources using satellite observations\nof atmospheric methane\nDaniel J. Jacob1, Daniel J. Varon1,2, Daniel H. Cusworth3,4, Philip E. Dennison5,\nChristian Frankenberg6,7, Ritesh Gautam8, Luis Guanter9,10, John Kelley11, Jason McKeever2,\nLesley E. Ott12, Benjamin Poulter12, Zhen Qu1, Andrew K. Thorpe7, John R. Worden7, and\nRiley M. Duren3,4,7\n1School of Engineering and Applied Sciences, Harvard University, Cambridge, 02138, USA\n2GHGSat, Inc., Montreal, H2W 1Y5, Canada\n3Arizona Institutes for Resilience, University of Arizona, Tucson, 85721, USA\n4Carbon Mapper, Pasadena, 91109, USA\n5Department of Geography, University of Utah, Salt Lake City, 84112, USA\n6Division of Geological and Planetary Sciences, California Institute

## 2. Data Format from the Data Search Agent

In [4]:
import json
from helper import parse_stac_items_to_collection_items
from data_types import SearchedSTACData, CollectionItem
from typing import List

with open("./data/stac_collection.json", "r") as f:
  stac_collection = json.load(f)

with open("./data/stac_items.json", "r") as f:
  stac_collection_items = json.load(f)

collection_items: List[CollectionItem] = parse_stac_items_to_collection_items(stac_collection_items, stac_collection)

In [5]:
collection_items

[CollectionItem(collection_id='blueflux-ghgflux-daygrid-v1', collection_description='This dataset includes daily gridded estimates of greenhouse gas fluxes (CO2 and CH4) across Southern Florida mangrove and marsh ecosystems from February 2000 to August 2024. Estimates were generated using flux data from airborne measurements during the BlueFlux project from 2022 to 2024 and regional eddy covariance towers since 2004, upscaled using satellite observations from the MODerate-resolution Imaging Spectroradiometer (MODIS) and machine learning modeling. The data has a spatial resolution of 500 m and are in units of micromoles of carbon dioxide per square meter per second (μmol CO2/m2/s) and nanomoles of methane per square meter per second (nmol CH4/m2/s). The source data and accompanying documentation can be found at https://doi.org/10.3334/ORNLDAAC/2404', collection_title='BlueFlux CO2 and Methane Fluxes for Southern Florida Wetlands', item_id='blueflux-ghgflux-daygrid-v1-2024-08-31', locati

## 3. Data Scount Agent 

### a. Relevant data filter component

In [6]:
# from relevant_data_filter_agent import get_relevant_data

# relevant_data: List[CollectionItem] = await get_relevant_data(api_key=api_key, literature_context=scraped_text, stac_data=collection_items)

In [7]:
# print("number of collection_items: ", len(collection_items), "\nnumber of relevant_data: ", len(relevant_data))

In [8]:
# json_relevant_data = [rd.model_dump_json() for rd in relevant_data]
# print(json_relevant_data)

The above are commented as these subagents are part of the Data Scout Agent already.

### b. Data relationship builder component

In [9]:
#NA

This subagent is part of the Data Scout Agent already. 

#### DATA SCOUT AGENT IMPLEMENTATION

In [10]:
from data_scout_agent import DataScoutAgent, DataScoutAgentInputSchema, DataScoutAgentOutputSchema, DataScoutAgentConfig

data_scout_input: DataScoutAgentInputSchema = DataScoutAgentInputSchema(
  literature=scraped_text,
  collection_items=collection_items
)
data_scout_agent_config: DataScoutAgentConfig = DataScoutAgentConfig(api_key=api_key)
data_scout_agent: DataScoutAgent = DataScoutAgent(config=data_scout_agent_config)
data_scout_agent_output: DataScoutAgentOutputSchema = await data_scout_agent.arun(data_scout_input)

/Users/thapa24-mbp/Devs/sandbox/accelerated-discovery/akd/configs/storyteller_prompts.py:224: SyntaxWarning: invalid escape sequence '\&'
  - [Background \& Prerequisites](#background--prerequisites)
11:40:59 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= gpt-4o; provider = openai
2025-12-09 11:40:59,393 - INFO - 
LiteLLM completion() model= gpt-4o; provider = openai


In [11]:
data_scout_agent_output

DataScoutAgentOutputSchema(relevant_collection=[])

In [12]:
print("Number of input collections items: ", len(data_scout_input.collection_items))
print("Number of relevant collections items: ", len(data_scout_agent_output.relevant_collection))

Number of input collections items:  10
Number of relevant collections items:  0


## 4. Script writer agent

### a. Script blueprint builder component

In [13]:
from script_blueprint_builder_agent import ScriptBlueprintBuilderAgent, ScriptBlueprintBuilderAgentConfig, ScriptBlueprintBuilderAgentInputSchema, ScriptBlueprintBuilderAgentOutputSchema

config = ScriptBlueprintBuilderAgentConfig(api_key=api_key)

script_blueprint_input: ScriptBlueprintBuilderAgentInputSchema = ScriptBlueprintBuilderAgentInputSchema(
  literature_text=scraped_text,
  collection_items=data_scout_agent_output.relevant_collection
)

script_blueprint_agent: ScriptBlueprintBuilderAgent = ScriptBlueprintBuilderAgent(config=config)

script_blueprint_output: ScriptBlueprintBuilderAgentOutputSchema = await script_blueprint_agent.arun(script_blueprint_input)

11:41:05 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= gpt-4o; provider = openai
2025-12-09 11:41:05,578 - INFO - 
LiteLLM completion() model= gpt-4o; provider = openai


In [14]:
script_blueprint_output.script_blueprint

"**Title:** Monitoring Methane Emissions from Space: A Satellite Overview\n\n**Target Audience:** General Public\n\n**Logline:** Understanding the potential and necessity of satellite technology in monitoring global methane emissions to bolster climate action initiatives.\n\n**Script Body:**\n\n**Introduction:**\n- **Context/Scene Setting:** Picture the Earth from above, a blue and green marble speckled with clouds and teeming with life. Yet, unseen to the naked eye, a potent greenhouse gas, methane, seeps into the atmosphere from various sources, contributing significantly to global warming. Scientists are harnessing the power of satellites to pinpoint and quantify these emissions across the globe.\n- **Visual Description:** Open with breathtaking clips of Earth from space, transitioning to time-lapsed imagery of sprawling forests, bustling oil fields, and wetlands.\n- **Narrative Text:** Methane is responsible for approximately 0.6°C of global warming, significantly impacting our cli

### b. Script builder component

In [15]:
from script_builder_agent import ScriptBuilderAgent, ScriptBuilderAgentConfig, ScriptBuilderAgentInputSchema, ScriptBuilderAgentOutputSchema

script_builder_config = ScriptBuilderAgentConfig(api_key=api_key)
script_builder_input = ScriptBuilderAgentInputSchema(
  narrative_blueprint=script_blueprint_output.script_blueprint
)
script_builder_agent: ScriptBuilderAgent = ScriptBuilderAgent(config=script_builder_config)
script_builder_output: ScriptBuilderAgentOutputSchema = await script_builder_agent.arun(script_builder_input)

11:41:25 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
2025-12-09 11:41:25,541 - INFO - 
LiteLLM completion() model= gpt-4o-mini; provider = openai


In [16]:
script_builder_output.story_draft

'<article>\n<h1>Monitoring Methane Emissions from Space: A Satellite Overview</h1>\n\n<section class="chapter">\n<h3>Introduction</h3>\n<p>Picture the Earth from above, a blue and green marble speckled with clouds and teeming with life. Yet, unseen to the naked eye, a potent greenhouse gas—methane—seeps into the atmosphere from various sources, contributing significantly to global warming. Scientists are harnessing the power of satellites to pinpoint and quantify these emissions across the globe.</p>\n<p>Open with breathtaking clips of Earth from space, transitioning to time-lapsed imagery of sprawling forests, bustling oil fields, and wetlands. Methane is responsible for approximately <strong>0.6°C</strong> of global warming, significantly impacting our climate. Satellite observations, such as those from GOSAT and TROPOMI, are becoming pivotal tools in managing and mitigating these emissions.</p>\n</section>\n\n<section class="chapter">\n<h3>The Satellite Fleet</h3>\n<p>High above the

### c. Script Critique

In [17]:
## Lets assume that we do not need critique.
## TODO: make this after data injection component

#### SCRIPT WRITER IMPLEMENTATION

In [18]:
# from script_writer_agent import ScriptWriterAgent, ScriptWriterAgentConfig, ScriptWriterAgentInputSchema, ScriptWriterAgentOutputSchema

# script_writer_config = ScriptWriterAgentConfig(api_key=api_key)

# script_writer_input: ScriptWriterAgentInputSchema = ScriptWriterAgentInputSchema(
#   literature_text=scraped_text,
#   collection_items=data_scout_agent_output.relevant_collection
# )
# script_writer_agent: ScriptWriterAgent = ScriptWriterAgent(config=script_writer_config)
# script_writer_output: ScriptWriterAgentOutputSchema = await script_writer_agent.arun(script_writer_input)

In [19]:
# script_writer_output.script

## 5. Data Injection Component

In [20]:
from data_injection_agent import DataInjectionAgent, DataInjectionAgentConfig, DataInjectionAgentInputSchema, DataInjectionAgentOutputSchema

data_injection_agent_config: DataInjectionAgentConfig = DataInjectionAgentConfig(api_key=api_key)

data_injection_agent_input: DataInjectionAgentInputSchema = DataInjectionAgentInputSchema(
  script=script_builder_output.story_draft,
  collection_items=data_scout_agent_output.relevant_collection
)

data_injection_agent: DataInjectionAgent = DataInjectionAgent(config=data_injection_agent_config)
data_injected_script: DataInjectionAgentOutputSchema = await data_injection_agent.arun(data_injection_agent_input)


11:41:45 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
2025-12-09 11:41:45,082 - INFO - 
LiteLLM completion() model= gpt-4o-mini; provider = openai


In [21]:
data_injected_script.script_with_data

'<article>\n<h1>Monitoring Methane Emissions from Space: A Satellite Overview</h1>\n\n<section class="chapter">\n<h3>Introduction</h3>\n<p>Picture the Earth from above, a blue and green marble speckled with clouds and teeming with life. Yet, unseen to the naked eye, a potent greenhouse gas—methane—seeps into the atmosphere from various sources, contributing significantly to global warming. Scientists are harnessing the power of satellites to pinpoint and quantify these emissions across the globe.</p>\n<p>Open with breathtaking clips of Earth from space, transitioning to time-lapsed imagery of sprawling forests, bustling oil fields, and wetlands. Methane is responsible for approximately <strong>0.6°C</strong> of global warming, significantly impacting our climate. Satellite observations, such as those from GOSAT and TROPOMI, are becoming pivotal tools in managing and mitigating these emissions.</p>\n</section>\n\n<section class="chapter">\n<h3>The Satellite Fleet</h3>\n<p>High above the

## 6. MDX Builder Component

In [22]:
from mdx_builder_agent import MDXBuilderAgent, MDXBuilderAgentConfig, MDXBuilderAgentInputSchema, MDXBuilderAgentOutputSchema

mdx_builder_config: MDXBuilderAgentConfig = MDXBuilderAgentConfig(api_key=api_key)

mdx_builder_input_schema: MDXBuilderAgentInputSchema = MDXBuilderAgentInputSchema(
    story_script=data_injected_script.script_with_data
  )

mdx_builder_agent: MDXBuilderAgent = MDXBuilderAgent(mdx_builder_config)
mdx_builder_output: MDXBuilderAgentOutputSchema = await mdx_builder_agent.arun(mdx_builder_input_schema)

mdx_story: str = mdx_builder_output.story_mdx

11:42:01 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= gpt-4o; provider = openai
2025-12-09 11:42:01,313 - INFO - 
LiteLLM completion() model= gpt-4o; provider = openai


In [23]:
mdx_story

"<Block>\n  <Prose>\n    ## Monitoring Methane Emissions from Space: A Satellite Overview\n  </Prose>\n</Block>\n\n<Block>\n  <Prose>\n    ### Introduction\n\n    Picture the Earth from above, a blue and green marble speckled with clouds and teeming with life. Yet, unseen to the naked eye, a potent greenhouse gas—methane—seeps into the atmosphere from various sources, contributing significantly to global warming. Scientists are harnessing the power of satellites to pinpoint and quantify these emissions across the globe.\n\n    Open with breathtaking clips of Earth from space, transitioning to time-lapsed imagery of sprawling forests, bustling oil fields, and wetlands. Methane is responsible for approximately **0.6°C** of global warming, significantly impacting our climate. Satellite observations, such as those from GOSAT and TROPOMI, are becoming pivotal tools in managing and mitigating these emissions.\n  </Prose>\n</Block>\n\n<Block>\n  <Prose>\n    ### The Satellite Fleet\n\n    Hig